# Apache Parquet - Rust

All 9 Rust examples from [docs/parquet.md](https://platob.github.io/yggdryl/parquet/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::io::Buffer;
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, Url};

// A non-null struct Field is the schema.
let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");

let arrow_schema = field.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2, 3])),
        Arc::new(StringArray::from(vec![Some("AAPL"), None, Some("MSFT")])),
    ],
)?;

let url = Url::from_str("file:///trades.parquet")?;
let mut media = Parquet::new(Buffer::new().with_media_type(url.media_type()));
media.write_batch_reader(arrow::batch_reader(arrow_schema, [batch.clone()]))?;

// Reading streams: one batch at a time, never one materialized table.
let read = media.read_batch_reader(None)?.collect::<Result<Vec<_>, _>>()?;
assert_eq!(read, [batch]);
assert_eq!(read[0].num_rows(), 3);

## Column pushdown

In [ ]:
use std::sync::Arc;

use arrow_array::{Float64Array, Int64Array, RecordBatch, RecordBatchReader, StringArray};
use yggdryl::arrow;
use yggdryl::io::Buffer;
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, MimeType};

let stored = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.required_field("symbol"),
    DataType::Float64.required_field("price"),
    DataType::Utf8.required_field("venue"),
])?
.required_field("row");
let arrow_schema = stored.to_arrow_schema()?;

let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2])),
        Arc::new(StringArray::from(vec!["AAPL", "MSFT"])),
        Arc::new(Float64Array::from(vec![1.5, 2.5])),
        Arc::new(StringArray::from(vec!["XNAS", "XNAS"])),
    ],
)?;

let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()));
media.write_batch_reader(arrow::batch_reader(arrow_schema, [batch]))?;

// Two of the four columns, named by a root Field of its own.
let wanted = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Float64.required_field("price"),
])?
.required_field("row");

let projected = media.read_batch_reader(Some(&wanted))?;
assert_eq!(projected.schema().fields().len(), 2);
let read = projected.collect::<Result<Vec<_>, _>>()?;
assert_eq!(read[0].num_columns(), 2);

// The file is unchanged: it still stores all four.
assert_eq!(media.read_schema()?.fields().len(), 4);

## Options

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::Buffer;
use yggdryl::parquet::{Parquet, ParquetOptions};
use yggdryl::{DataType, Level, MimeType};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = field.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from((0..1_000).collect::<Vec<i64>>()))],
)?;

// Parquet's own settings and the shared ones are flat fields on one struct.
let options = ParquetOptions::new()
    .with_max_row_group_size(4_096)
    .with_key_value("iceberg.schema-id", "7")
    .with_batch_size(256)
    .with_root_name("trade");

assert_eq!(options.max_row_group_size, 4_096);
assert_eq!(
    options.key_value_metadata,
    [("iceberg.schema-id".to_owned(), "7".to_owned())]
);
assert_eq!(options.batch_size(), Some(256));
assert_eq!(options.root_name(), "trade");
assert!(!options.safe());

// Unused here: Parquet compresses pages itself.
assert_eq!(options.level, Level::DEFAULT);

let mut media =
    Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into())).with_options(options);
media.write_batch_reader(arrow::batch_reader(arrow_schema, [batch]))?;

// batch_size bounds the reader, so no batch holds all 1,000 rows.
let rows: Vec<usize> = media
    .read_batch_reader(None)?
    .collect::<Result<Vec<_>, _>>()?
    .iter()
    .map(arrow_array::RecordBatch::num_rows)
    .collect();
assert_eq!(rows.iter().sum::<usize>(), 1_000);
assert!(rows.iter().all(|count| *count <= 256), "{rows:?}");

// The root name names the Field recovered from the footer.
assert_eq!(media.read_field()?.name(), "trade");

// A declared schema is returned as-is, so an empty handle answers without a footer.
let declared =
    Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into())).with_schema(field.clone());
assert_eq!(declared.read_field()?, field);

## Compression

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use parquet::basic::Compression;
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::parquet::{Parquet, ParquetOptions};
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");

let ids: Vec<i64> = (0..4_000).collect();
let symbols: Vec<Option<&str>> = ids.iter().map(|_| Some("AAPL")).collect();
let arrow_schema = field.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(ids)),
        Arc::new(StringArray::from(symbols)),
    ],
)?;

let mut sizes = Vec::new();
for compression in [
    Compression::UNCOMPRESSED,
    Compression::SNAPPY,
    Compression::ZSTD(Default::default()),
] {
    // One batch per read, so the comparison is not split by the default bound.
    let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()))
        .with_options(
            ParquetOptions::new()
                .with_compression(compression)
                .with_batch_size(batch.num_rows()),
        );
    media.write_batch_reader(arrow::batch_reader(Arc::clone(&arrow_schema), [batch.clone()]))?;

    // Nothing on the read side names the compression: the footer records it.
    let read = media.read_batch_reader(None)?.collect::<Result<Vec<_>, _>>()?;
    assert_eq!(read, [batch.clone()], "{compression:?}");
    sizes.push(media.handle().size());
}

assert!(sizes[0] > sizes[1] && sizes[0] > sizes[2], "{sizes:?}");

## Coded handles are rejected

In [ ]:
use arrow_array::RecordBatch;
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, Url};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

// The name declares gzip over the Parquet file.
let url = Url::from_str("file:///trades.parquet.gz")?;
let mut media = Parquet::new(Buffer::new().with_media_type(url.media_type()));

let empty = arrow::batch_reader(field.to_arrow_schema()?, std::iter::empty::<RecordBatch>());
let message = media.write_batch_reader(empty).unwrap_err().to_string();
assert!(message.contains("parquet compresses"), "{message}");
assert!(message.contains("ParquetOptions::compression"), "{message}");

// Nothing was published.
assert!(media.handle().is_empty());

## Field identifiers

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::io::Buffer;
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([
    DataType::Int64.required_field("id").with_parquet_field_id(1),
    DataType::Utf8.nullable_field("symbol").with_parquet_field_id(2),
])?
.required_field("row");

let arrow_schema = field.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1])),
        Arc::new(StringArray::from(vec![Some("AAPL")])),
    ],
)?;

let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()));
media.write_batch_reader(arrow::batch_reader(arrow_schema, [batch]))?;

// The ids went into the file, so the Arrow schema carries them back.
let schema = media.read_schema()?;
assert_eq!(
    schema.field(0).metadata().get("PARQUET:field_id"),
    Some(&"1".to_owned())
);

// And the recovered Field answers by id rather than by position.
let recovered = media.read_field()?;
assert_eq!(recovered.fields()[0].parquet_field_id()?, Some(1));
assert_eq!(recovered.fields()[1].parquet_field_id()?, Some(2));

## Footer statistics

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::io::Buffer;
use yggdryl::parquet::{Parquet, ParquetOptions};
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");

let ids: Vec<i64> = (0..2_048).collect();
let symbols: Vec<Option<&str>> = ids
    .iter()
    .map(|index| (index % 2 == 0).then_some("AAPL"))
    .collect();
let arrow_schema = field.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(ids)),
        Arc::new(StringArray::from(symbols)),
    ],
)?;

let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into())).with_options(
    ParquetOptions::new()
        .with_max_row_group_size(512)
        .with_key_value("iceberg.schema-id", "7"),
);
media.write_batch_reader(arrow::batch_reader(arrow_schema, [batch]))?;

let statistics = media.read_statistics()?;
assert_eq!(statistics.num_rows, 2_048);
assert_eq!(statistics.row_groups.len(), 4);
assert!(statistics.created_by.is_some());
assert!(
    statistics
        .key_value_metadata
        .iter()
        .any(|(key, value)| key == "iceberg.schema-id" && value == "7"),
    "{:?}",
    statistics.key_value_metadata
);

// Null counts are summed over every row group; an unknown path is None, not zero.
assert_eq!(statistics.null_count("symbol"), Some(1_024));
assert_eq!(statistics.null_count("id"), Some(0));
assert_eq!(statistics.null_count("absent"), None);

// One offset per row group, ascending.
let offsets = statistics.split_offsets();
assert_eq!(offsets.len(), 4);
assert!(offsets.windows(2).all(|pair| pair[0] < pair[1]), "{offsets:?}");

let first = &statistics.row_groups[0];
assert_eq!(first.num_rows, 512);
assert!(first.compressed_size > 0);
assert!(
    first
        .columns
        .iter()
        .any(|column| column.path == "id" && column.min_bytes.is_some())
);

## The handle underneath

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::parquet::{self, Parquet, ParquetOptions};
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = field.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from(vec![1, 2]))],
)?;

// The free functions take a handle and options; nothing is bound.
let options = ParquetOptions::new();
let mut handle = Buffer::new().with_media_type(MimeType::PARQUET.into());
parquet::write_batch_reader(
    &mut handle,
    arrow::batch_reader(arrow_schema, [batch]),
    &options,
)?;

assert_eq!(parquet::read_schema(&handle)?.fields().len(), 1);
assert_eq!(parquet::read_field(&handle, &options)?.name(), "row");
assert_eq!(parquet::read_batch_reader(&handle, None, &options)?.count(), 1);
assert_eq!(parquet::read_statistics(&handle)?.num_rows, 2);

// A Parquet is also the bytes it encodes, magic bytes included.
let mut media = Parquet::new(handle);
assert_eq!(media.read_range(0, 4)?, *b"PAR1");

// open caches the footer, close releases it.
assert!(!media.is_open());
media.open()?;
assert!(media.is_open());
assert_eq!(media.read_statistics()?.num_rows, 2);
media.close()?;
assert!(!media.is_open());

In [ ]:
use arrow_array::{RecordBatch, RecordBatchReader};
use yggdryl::arrow;
use yggdryl::io::Buffer;
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

// Nothing has been written, so there is nothing to read.
let empty = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()))
    .with_schema(field.clone());
assert_eq!(empty.read_batch_reader(None)?.count(), 0);
assert_eq!(empty.read_batch_reader(None)?.schema().fields().len(), 1);

// An empty write still publishes a readable file with the schema in its footer.
let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()));
media.write_batch_reader(arrow::batch_reader(
    field.to_arrow_schema()?,
    std::iter::empty::<RecordBatch>(),
))?;
assert_eq!(media.read_batch_reader(None)?.count(), 0);
assert_eq!(media.read_schema()?.fields().len(), 1);
assert_eq!(media.read_statistics()?.num_rows, 0);